In [1]:
!pip install -q \
langchain==0.3.26 \
langchain-community==0.3.27 \
langchain-huggingface==0.3.1 \
sentence-transformers==5.0.0 \
faiss-cpu==1.12.0 \
transformers==4.55.4 \
accelerate==1.10.1 \
peft==0.17.1 \
gradio==5.43.1 \
pypdf==5.9.0 \
rank-bm25==0.2.2 \
duckduckgo-search==8.1.1

In [2]:
import torch
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from peft import PeftModel

from langchain_community.document_loaders import PyPDFLoader

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import BM25Retriever

from langchain.retrievers import EnsembleRetriever

from langchain.retrievers import ParentDocumentRetriever

from langchain.storage import InMemoryStore

from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain.retrievers.document_compressors import CrossEncoderReranker

from langchain.retrievers import ContextualCompressionRetriever

from langchain_community.tools import DuckDuckGoSearchRun

In [3]:
!gdown --folder "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql"

Retrieving folder contents
Processing file 1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU PP Nomor 5 Tahun 2021.pdf
Processing file 1trvqHE72Anu8MhsNvvYJHieWqbao2BCy PP Nomor 35 Tahun 2021.pdf
Processing file 1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT PP Nomor 51 Tahun 2023.pdf
Processing file 1jf5f9ZHF2tzcK7Mm7eHCkDEtVntQ-y1s UU Nomor 6 Tahun 2023.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1dqrM0fgltQb1Ot3uMTVXZq3wTCNEyPoU
To: /content/Document RAG/PP Nomor 5 Tahun 2021.pdf
100% 17.1M/17.1M [00:00<00:00, 23.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1trvqHE72Anu8MhsNvvYJHieWqbao2BCy
To: /content/Document RAG/PP Nomor 35 Tahun 2021.pdf
100% 2.52M/2.52M [00:00<00:00, 16.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wXSVlaS_Nk4Yt9kWm6kkcYS-XY_-XyfT
To: /content/Document RAG/PP Nomor 51 Tahun 2023.pdf
100% 2.77M/2.77M [00:00<00:00, 18.6MB/s]
Downloading...
From: ht

In [4]:
import os

os.listdir()

['.config', '.gradio', 'Document RAG', 'sample_data']

In [5]:
import glob

pdf_files = glob.glob(
    "Document RAG/**/*.pdf",
    recursive=True
)

pdf_files

['Document RAG/PP Nomor 35 Tahun 2021.pdf',
 'Document RAG/PP Nomor 5 Tahun 2021.pdf',
 'Document RAG/UU Nomor 6 Tahun 2023.pdf',
 'Document RAG/PP Nomor 51 Tahun 2023.pdf']

In [6]:
from langchain_community.document_loaders import PyPDFLoader

all_docs = []

for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()

    all_docs.extend(docs)

print(len(all_docs))

1949


In [7]:
for doc in all_docs:

    # nama file pdf
    source_path = doc.metadata.get("source", "")

    filename = os.path.basename(source_path)

    # metadata tambahan
    doc.metadata["category"] = "legal_document"
    doc.metadata["filename"] = filename

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [10]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

In [11]:
vectorstore = FAISS.from_texts(
    ["init"],
    embedding_model
)

In [12]:
# hapus dummy doc
vectorstore.delete([vectorstore.index_to_docstore_id[0]])

True

In [13]:
store = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_retriever.add_documents(all_docs)

In [14]:
bm25_retriever = BM25Retriever.from_documents(all_docs)

bm25_retriever.k = 5

In [15]:
vector_retriever = parent_retriever

In [16]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        parent_retriever
    ],
    weights=[0.4, 0.6]
)

In [17]:
model = AutoModelForCausalLM.from_pretrained(
    "rafiqiraihan/qwen-rag-indonesia",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "rafiqiraihan/qwen-rag-indonesia"
)

In [18]:
model.config.pad_token_id = tokenizer.eos_token_id
model.config.eos_token_id = tokenizer.eos_token_id

In [25]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

README.md: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [26]:
search = DuckDuckGoSearchRun()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [27]:
def generate_response(messages, max_new_tokens=128):

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.1,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [28]:
def hyde_query(question):

    messages = [
        {
            "role": "system",
            "content": """
Buat jawaban hipotetis singkat
untuk membantu pencarian dokumen.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    response = generate_response(
        messages,
        max_new_tokens=64
    )

    return response

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [29]:
def rag_pipeline(question):

    # ====================================
    # HYDE
    # ====================================

    hypothetical_answer = hyde_query(question)

    hyde_query_text = f"""
    Pertanyaan:
    {question}

    Jawaban hipotetis:
    {hypothetical_answer}
    """


    # ====================================
    # RETRIEVE DOCUMENTS
    # ====================================

    retrieved_docs = ensemble_retriever.invoke(
        hyde_query_text
    )

    if not retrieved_docs:
        return "Tidak ada dokumen ditemukan."


    # ====================================
    # METADATA FILTERING
    # ====================================

    retrieved_docs = [
        doc for doc in retrieved_docs
        if doc.metadata.get("category") == "legal_document"
    ]


    print("\n===== RETRIEVED DOCS =====")

    for i, doc in enumerate(retrieved_docs):
        print(f"\nDoc {i+1}")
        print(doc.metadata)
        print(doc.page_content[:300])


    # ====================================
    # RERANKING
    # ====================================

    pairs = [
        (question, doc.page_content)
        for doc in retrieved_docs
    ]

    scores = cross_encoder.score(pairs)

    scored_docs = sorted(
        zip(scores, retrieved_docs),
        key=lambda x: x[0],
        reverse=True
    )


    # ====================================
    # TOP DOCS
    # ====================================

    top_docs = [
        doc for score, doc
        in scored_docs[:3]
    ]

    top_score = scored_docs[0][0]

    print("Top Score:", top_score)


    # ====================================
    # FALLBACK INTERNET SEARCH
    # ====================================

    THRESHOLD = 0.0

    if top_score < THRESHOLD:

        internet_result = search.run(question)

        return f"""
Jawaban dari Internet:

{internet_result}
"""


    # ====================================
    # BUILD CONTEXT
    # ====================================

    context = "\n\n".join([
        doc.page_content[:400]
        for doc in top_docs
    ])


    # ====================================
    # CITATIONS
    # ====================================

    citations = "\n".join([
        f"- {doc.metadata.get('filename')} halaman {doc.metadata.get('page')}"
        for doc in top_docs
    ])


    # ====================================
    # PROMPT
    # ====================================

    messages = [
        {
            "role": "system",
            "content": """
Anda adalah AI assistant untuk question answering dokumen hukum.

Jawab pertanyaan SECARA SINGKAT dan JELAS.

Jawab HANYA berdasarkan context.

Jangan mengulang kalimat.

Jangan menambahkan informasi di luar context.

Jika jawaban tidak ditemukan dalam context,
katakan:
'Informasi tidak ditemukan dalam dokumen.'
"""
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{question}
"""
        }
    ]


    # ====================================
    # DEBUG DOCS
    # ====================================

    for score, doc in scored_docs[:3]:

        print("\n====================")
        print("Score:", score)
        print(doc.page_content[:500])


    # ====================================
    # GENERATION
    # ====================================

    result = generate_response(
        messages,
        max_new_tokens=128
    )


    # ====================================
    # CLEAN OUTPUT
    # ====================================

    answer = result.split("assistant")[-1].strip()


    # ====================================
    # FINAL OUTPUT
    # ====================================

    final_answer = f"""
{answer}

Top Relevance Score: {top_score:.2f}

Sumber:
{citations}
"""

    return final_answer

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [30]:
demo = gr.Interface(
    fn=rag_pipeline,
    inputs="text",
    outputs="text",
    title="Legal RAG System"
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


* Running on public URL: https://8464a050d0b6cf4455.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
